# Session 13 · Exercises

Work in pairs. `help()`, `dir()` and the lecture notebooks answer most questions. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are solved without a laptop first; write the answer into the cell below the task.

Run the setup cell of a part before its *Code* tasks. The data come from `case-study/data/` (prepare them with `case-study/prepare_data.py`).

*Author: course team, licence CC BY 4.0.*

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Part 1 · Text as data

In [ ]:
# setup for the Code tasks of part 1
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

decisions = pd.read_parquet("case-study/data/train_sample.parquet")
texts = decisions["description"]
print(decisions.shape)

### 1.1 · "Dog bites man" against "man bites dog"

*Paper, 5 min.*

> Write both as bag-of-words vectors.

*Source:* [Jurafsky & Martin, *Speech and Language Processing*, Appendix B: the bag-of-words assumption](https://web.stanford.edu/~jurafsky/slp3/B.pdf)

*Your answer:*

### 1.2 · TF-IDF by hand

*Paper, 15 min.*

> Three short product descriptions; which word gets the highest weight in each?
>
> - d1 = "plastic toy car with plastic wheels"
> - d2 = "wooden toy train"
> - d3 = "plastic storage box with lid"
>
> Use the textbook formula tf × ln(N / df) with raw counts as tf and no normalisation. Then check your result with `TfidfVectorizer(smooth_idf=False, norm=None)`. Why does scikit-learn's version give different numbers?

*Source:* [Jurafsky & Martin, *Speech and Language Processing*, Chapter 11: tf-idf](https://web.stanford.edu/~jurafsky/slp3/11.pdf); original: [Spärck Jones (1972), *Journal of Documentation*](https://doi.org/10.1108/eb026526)

*Your answer:*

### 1.3 · Predict the stemmer

*Predict, 5 min.*

> What does the Porter stemmer make of *university, universe, universal, generous, generate, general, organization, organ, policy, police, batteries*?
>
> Write your prediction down, then check it with `from nltk.stem import PorterStemmer`.

*Source:* [Porter, The Porter stemming algorithm](https://tartarus.org/martin/PorterStemmer/)

*Your answer:*

### 1.4 · Remove the stop words

*Predict, 3 min.*

> "not compatible with" loses *not*. What happens to the meaning?
>
> Check which of the words "not", "no", "without", "nor" and "ohne" are in scikit-learn's `ENGLISH_STOP_WORDS`.

*Source:* original exercise for this course; background: [Jurafsky & Martin, *Speech and Language Processing*, Appendix B: negation in sentiment analysis](https://web.stanford.edu/~jurafsky/slp3/B.pdf)

*Your answer:*

### 1.5 · Guess the language

*Class, then Code, 20 min.*

> The cell below shows ten EBTI descriptions without the language label. Guess each language from clues such as *sch*, *ij*, *ção*, *ł*. Then build a language detector from character trigram counts in about 15 lines: one trigram profile per language from the training descriptions, and for a new text the language whose profile is most similar. How many of the ten does it get right?

*Source:* [Cavnar & Trenkle (1994), "N-gram-based text categorization"](https://www.let.rug.nl/~vannoord/TextCat/textcat.pdf)

In [ ]:
# ten descriptions in ten languages, drawn with a fixed seed; the labels stay hidden in `secret`
quiz_languages = ["de", "fr", "nl", "pl", "cs", "es", "sv", "it", "hu", "da"]
quiz = (decisions[decisions["language"].isin(quiz_languages)]
        .groupby("language").sample(1, random_state=13)
        .sample(frac=1, random_state=13))
secret = quiz["language"].tolist()
for i, text in enumerate(quiz["description"], start=1):
    print(i, " ".join(text.split())[:160])

*Your answer:*

### 1.6 · A text statistic of your own

*Code, 15 min.*

> The lecture computed four statistics (length, digits, upper-case share, per-cent sign). Write one more that you expect to carry information about the goods, for example "mentions a measurement unit (cm, mm, kg, V, W)" or "number of commas". Which five chapters have the highest share of descriptions with your feature? Then add it to the four lecture statistics and refit the decision tree of the lecture (train 2017–2021, validate 2022–2023): does the chapter accuracy change?

In [ ]:
# the four statistics of the lecture, as a starting point
def text_stats(text):
    t = text.fillna("")
    letters = t.map(lambda s: sum(ch.isalpha() for ch in s)).clip(lower=1)
    return pd.DataFrame({
        "n_chars": t.str.len(),
        "n_digits": t.map(lambda s: sum(ch.isdigit() for ch in s)),
        "upper_share": t.map(lambda s: sum(ch.isupper() for ch in s)) / letters,
        "has_percent": t.str.contains("%", regex=False).astype(int),
    }, index=text.index)


stats = text_stats(texts)
year = decisions["start_date"].dt.year
past, later = year <= 2021, year >= 2022

### 1.7 · Shouting titles

*Code, 15 min.*

> Do titles in capitals, with emojis or with "!!!" have different prices? Build three title features and measure them.
>
> The data are the Airbnb Berlin listings of Sessions 1–12 (titles in the column `name`). Compare the median price of listings with and without each feature, and say how many listings have it. *Stretch:* add the three features to the price model of Session 9 with host-grouped cross-validation.

*Source:* original exercise for this course; background: [pandas user guide: Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)

In [ ]:
# Airbnb listings with a nightly price, short stays only (as in Sessions 7 and 9)
listings = pd.read_parquet("case-study/data/airbnb/listings.parquet")
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
titles = bnb["name"].fillna("")
print(len(bnb), titles.head(3).tolist())

### 1.8 · Kraftfahrzeughaftpflichtversicherung

*Discussion, 5 min.*

> What does a word-level bag of words do with German compounds?
>
> Find three compounds in the case-study vocabulary that end in "schuhe" or "spielzeug" and say which columns they share with "Schuhe" or "Spielzeug" under word tokens and under `char_wb` 4-grams.

*Source:* original exercise for this course; background: [Bojanowski et al. (2017), "Enriching word vectors with subword information"](https://arxiv.org/abs/1607.04606)

*Your answer:*

### 1.9 · Edit distance by hand (*Stretch*)

*Paper, 15 min.*

> Fill in the dynamic-programming table that turns *intention* into *execution*; insertion, deletion and substitution each cost 1. Then repeat with substitution costing 2.

*Source:* [Jurafsky & Martin, *Speech and Language Processing*, Chapter 2: minimum edit distance](https://web.stanford.edu/~jurafsky/slp3/2.pdf)

*Your answer:*

## Part 2 · Text classification and error analysis

In [ ]:
# setup for the Code tasks of part 2 (about 30 seconds): the word model of the lecture
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline

decisions = pd.read_parquet("case-study/data/train_sample.parquet")
nomenclature = pd.read_parquet("case-study/data/nomenclature.parquet").set_index("heading")
year = decisions["start_date"].dt.year
train, valid = decisions[year <= 2021], decisions[year >= 2022]          # validation by time
model = make_pipeline(TfidfVectorizer(min_df=2, sublinear_tf=True),
                      SGDClassifier(loss="hinge", alpha=1e-5, max_iter=20, tol=None,
                                    random_state=0, n_jobs=-1))
model.fit(train["description"], train["heading"])
pred = model.predict(valid["description"])
val = valid.assign(pred=pred)
print(round(accuracy_score(valid["heading"], pred), 3))

### 2.1 · Naive Bayes by hand

*Paper, 15 min.*

> Training data: negative "just plain boring", "entirely predictable and lacks energy", "no surprises and very few laughs"; positive "very powerful", "the most fun film of the summer". With add-one smoothing (vocabulary of 20 words), classify "predictable with no fun".
>
> *Stretch:* what happens without smoothing?

*Source:* [Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft), Appendix B: Naive Bayes, worked example](https://web.stanford.edu/~jurafsky/slp3/B.pdf)

*Your answer:*

### 2.2 · Humans against the baseline

*Class, 15 min.*

> Classify ten descriptions into HS chapters by hand. How does the class compare with TF-IDF?
>
> The cell below shows ten English descriptions from the validation years and the list of chapters. Write down one two-digit chapter per description (the class may vote). Then compare your answers with the true chapters and with the chapters of the model's predictions (`val["pred"].str[:2]`).

*Source:* original exercise for this course; background: [Karpathy (2014), "What I learned from competing against a ConvNet on ImageNet"](http://karpathy.github.io/2014/09/02/what-i-learned-from-competing-against-a-convnet-on-imagenet/)

In [ ]:
ten = val[val["language"] == "en"].sample(10, random_state=7)
for i, text in enumerate(ten["description"], start=1):
    print(i, " ".join(text.split())[:300], "\n")
chapters = nomenclature.drop_duplicates("chapter").set_index("chapter")["chapter_description"].str[:70]
print(chapters.to_string())

*Your answer:*

### 2.3 · The fallback heading of chapter 39

*Code, 15 min.*

> The lecture read the confusion matrix of the footwear headings. Do the same for the plastic articles 3923 (containers, closures), 3924 (tableware, household articles), 3925 (builders' ware) and 3926 (other articles of plastics). Which heading acts as the model's fallback? Compute precision and recall of 3926 from the matrix and check them with `classification_report`.

### 2.4 · Ten errors in another language

*Pairs, 15 min.*

> Draw ten misclassified **French** (or Polish) decisions from the validation years with `random_state=2`. Read each one with its English keywords and the English texts of the true and the predicted heading, and give each one category from the lecture's list (borderline, material versus function, rare language, technical, set, new heading, unclear). Does the distribution differ from the lecture's 20 errors?

### 2.5 · What does the model rely on for two other headings?

*Code, 10 min.*

> List the twelve n-grams with the largest coefficients for heading 2106 (food preparations not elsewhere specified) and 9405 (luminaires and lighting fittings). How many of them are numbers, how many are words in German, French or English? Would you trust the model for a Greek description of a lamp?

### 2.6 · Humans write the classifier

*Paper, then Class, 10 min.*

> In two minutes, write five words that signal a positive film review and five that signal a negative one. A review is classified by counting hits; ties are guessed. Predict your accuracy.

*Source:* [Pang, Lee & Vaithyanathan (2002), "Thumbs up? Sentiment classification using machine learning techniques", EMNLP](https://aclanthology.org/W02-1011/)

*Your answer:*

### 2.7 · Character n-grams for the rare languages (*Stretch*)

*Code, 20 min.*

> Fit the character model of the lecture (`TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=300_000)` with the same linear SVM, about two minutes) and compare its validation accuracy with the word model **per language**. Do character n-grams help the languages with fewer than 500 training decisions more than German and French? How many validation decisions does each language have, and how far would you trust the per-language numbers?